A test to see which number is more suitable for set_num_threads

In [1]:
# import torch
# import time
#
# def benchmark_threads(num_threads, matrix_size=4000, iterations=20):
#     # Set the number of threads for this test run
#     torch.set_num_threads(num_threads)
#
#     # Create two massive random matrices to simulate deep learning workloads
#     A = torch.randn(matrix_size, matrix_size)
#     B = torch.randn(matrix_size, matrix_size)
#
#     # Warmup round (gets the CPU caches loaded and ready)
#     _ = torch.matmul(A, B)
#
#     # Start the actual timer
#     start_time = time.time()
#
#     # Run the heavy math multiple times
#     for _ in range(iterations):
#         _ = torch.matmul(A, B)
#
#     end_time = time.time()
#
#     print(f"Total time with {num_threads} threads: {end_time - start_time:.4f} seconds")
#
# print("--- PyTorch CPU Thread Benchmark ---")
# # Test with 4 threads
# benchmark_threads(4)
#
# # Test with 8 threads
# benchmark_threads(8)

--- PyTorch CPU Thread Benchmark ---
Total time with 4 threads: 13.5976 seconds
Total time with 8 threads: 8.8375 seconds


8 threads is faster than 4 threads, so we will use 8 threads for the rest of the code

Cell 1: Imports and Device Setup

In [33]:
import os
from statistics import mean

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import time

# Explicitly tell PyTorch to utilize your 8 CPU cores for matrix math
torch.set_num_threads(8)
print(f"PyTorch is now using {torch.get_num_threads()} threads.")

# Check for GPU availability to drastically speed up training
# ! cuda is available only on nvidia gpu
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Executing on device: {device}")

PyTorch is now using 8 threads.
Executing on device: cpu


Cell 1.5: Changing the data format from .npz to .pt for faster loading (one-time setup)

In [18]:
# def normalize_and_save():
#     input_dir = "./16QAMdata/"
#     output_dir = "./16QAMpt/"
#     os.makedirs(output_dir, exist_ok=True)
#
#     print("Converting .npz to normalized .pt files...")
#     for i in range(100):
#         print(f"Processing file {i + 1}/100: 16qam_tx_rx_32_part_{i:03d}.npz")
#         # 1. Load the slow .npz file
#         data = np.load(os.path.join(input_dir, f"16qam_tx_rx_32_part_{i:03d}.npz"))
#         X_raw = data['tx']
#         Y_raw = data['rx']
#
#         # 2. Extract Real and Imaginary
#         for part_idx, part_name in enumerate(['real', 'imag']):
#             X_batch = torch.tensor(X_raw[part_idx, :, :], dtype=torch.float32)
#             Y_batch = torch.tensor(Y_raw[part_idx, :, :], dtype=torch.float32)
#
#             # 3. Normalize once and for all
#             X_min, X_max = X_batch.min(), X_batch.max()
#             Y_min, Y_max = Y_batch.min(), Y_batch.max()
#
#             X_norm = 2.0 * ((X_batch - X_min) / (X_max - X_min)) - 1.0 if X_max != X_min else X_batch
#             Y_norm = 2.0 * ((Y_batch - Y_min) / (Y_max - Y_min)) - 1.0 if Y_max != Y_min else Y_batch
#
#             # 4. Save as a hyper-fast PyTorch file
#             torch.save((X_norm, Y_norm), os.path.join(output_dir, f"16qam_tx_rx_32_part_{i:03d}.pt"))
#
#     print("Done! You never have to do this math again.")
#
# normalize_and_save()

Cell 2: Custom Activation Function

In [19]:
class TriangularActivation(nn.Module):
    """
    Implements the triangular activation function used in the paper's hidden layers.
    Mathematically equivalent to MATLAB's 'tribas': f(x) = max(1 - |x|, 0)
    """
    def forward(self, x):
        return torch.clamp(1.0 - torch.abs(x), min=0.0)

Cell 3: Highly Optimized Dataset Class

This is faster as it works with .pt directly without conversion

In [20]:
class FastOFDMDataset(Dataset):
    def __init__(self, pt_folder_path, part='real'):
        self.folder_path = pt_folder_path
        self.part = part

    def __len__(self):
        return 100 # Assuming exactly 100 pre-computed .pt files

    def __getitem__(self, idx):
        # Instantly loads the pre-normalized tensors directly into memory!
        file_path = os.path.join(self.folder_path, f"16qam_tx_rx_32_part_{idx:03d}.pt")
        X_norm, Y_norm = torch.load(file_path, weights_only=True)
        return X_norm, Y_norm

In [21]:
# class OFDMDataset(Dataset):
#     def __init__(self, file_list, part='real'):
#         self.file_list = file_list
#         self.part = part
#
#     def __len__(self):
#         return len(self.file_list)
#
#     # @staticmethod
#     # def _scale_pm1(x):
#     #     # scale to [-1,1] without sklearn overhead
#     #     x_min = x.min(axis=0, keepdims=True)
#     #     x_max = x.max(axis=0, keepdims=True)
#     #     denom = np.where((x_max - x_min) == 0, 1.0, (x_max - x_min))
#     #     return 2.0 * (x - x_min) / denom - 1.0
#     #
#     # def __getitem__(self, idx):
#     #     data = np.load(self.file_list[idx])
#     #     X_raw = data["tx"]
#     #     Y_raw = data["rx"]
#     #     part_idx = 0 if self.part == "real" else 1
#     #     X_batch = X_raw[part_idx]
#     #     Y_batch = Y_raw[part_idx]
#     #     X_norm = self._scale_pm1(X_batch)
#     #     Y_norm = self._scale_pm1(Y_batch)
#     #     return torch.tensor(X_norm, dtype=torch.float32), torch.tensor(Y_norm, dtype=torch.float32)
#
#     def __getitem__(self, idx):
#         # 1. Load the specific .npz file
#         file_path = self.file_list[idx]
#         data = np.load(file_path)
#
#         # Extract input (original OFDM) and target (ICF) data
#         X_raw = data['tx']
#         Y_raw = data['rx']
#
#         # 2. Extract Real (0) or Imaginary (1) part
#         part_idx = 0 if self.part == 'real' else 1
#         X_batch = torch.tensor(X_raw[part_idx, :, :], dtype=torch.float32)
#         Y_batch = torch.tensor(Y_raw[part_idx, :, :], dtype=torch.float32)
#
#         # 3. Native PyTorch Normalization to strictly [-1, 1]
#         X_min, X_max = X_batch.min(), X_batch.max()
#         Y_min, Y_max = Y_batch.min(), Y_batch.max()
#
#         # Avoid division by zero in case of an empty/flat signal
#         if (X_max - X_min) != 0:
#             X_norm = 2.0 * ((X_batch - X_min) / (X_max - X_min)) - 1.0
#         else:
#             X_norm = X_batch
#
#         if (Y_max - Y_min) != 0:
#             Y_norm = 2.0 * ((Y_batch - Y_min) / (Y_max - Y_min)) - 1.0
#         else:
#             Y_norm = Y_batch
#
#         return X_norm, Y_norm

Cell 4: Neural Network Architecture

In [22]:
class NNICFMapper(nn.Module):
    def __init__(self, input_size):
        super(NNICFMapper, self).__init__()
        # First hidden layer: 2 neurons
        self.hidden1 = nn.Linear(input_size, 2)
        # Second hidden layer: 1 neuron
        self.hidden2 = nn.Linear(2, 1)
        # Output layer: Maps back to the original subcarrier points
        self.output = nn.Linear(1, input_size)
        # Custom triangular activation
        self.activation = TriangularActivation()

    def forward(self, x):
        x = self.activation(self.hidden1(x))
        x = self.activation(self.hidden2(x))
        x = self.output(x) # Standard linear output
        return x

Cell 5: Initialization and DataLoaders

In [25]:
# --- Configuration ---
# Update this path to where your 100 files are saved!
file_directory = "./16QAMpt/"
all_files = [os.path.join(file_directory, f"16qam_tx_rx_32_part_{i:03d}.pt") for i in range(100)]

# Initialize DataLoaders with asynchronous loading (num_workers) and rapid memory transfer
train_dataset_real = FastOFDMDataset(file_directory, part='real')
train_loader_real = DataLoader(
    train_dataset_real,
    batch_size=None, # batch_size is None because the dataset returns a full batch of 10,000
    shuffle=True,
    num_workers=0,   # Adjust between 2-4 based on your CPU cores
    pin_memory=torch.cuda.is_available()
)

train_dataset_imag = FastOFDMDataset(file_directory, part='imag')
train_loader_imag = DataLoader(
    train_dataset_imag,
    batch_size=None,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

# 1024 features based on N=256 and oversampling J=4
input_features = 1024

# Initialize models and send them to the GPU/device
Mod_Re_NN = NNICFMapper(input_features).to(device)
Mod_Im_NN = NNICFMapper(input_features).to(device)

# Standard Mean Squared Error loss and Adam optimizer
criterion = nn.MSELoss()
optimizer_real = optim.Adam(Mod_Re_NN.parameters(), lr=0.001)
optimizer_imag = optim.Adam(Mod_Im_NN.parameters(), lr=0.001)

This took the following times

In [36]:
real_time = [10.91, 10.85, 10.63, 11.44, 11.77, 14.69, 13.13, 11.39, 10.53, 10.59, 10.47, 10.33, 10.22, 10.33, 10.14, 10.35, 10.11, 10.02, 10.17, 10.29, 10.02, 10.10, 10.10, 9.97, 10.13, 10.14, 10.09, 10.32, 10.25, 10.04, 10.19, 10.17, 10.08, 10.02, 10.14, 10.27, 10.23, 10.25, 9.97, 11.72, 10.96, 10.29, 10.45, 10.43, 10.38, 10.30, 10.06, 10.23, 10.30, 10.10, 10.81, 10.80, 10.60, 10.21, 10.23, 10.21, 10.19, 10.35, 10.26, 10.11, 10.42, 10.29, 10.64, 10.67, 10.44, 10.37, 10.13, 10.09, 10.33, 10.46, 11.33, 10.62, 10.78, 10.42, 10.71, 10.54, 10.02, 10.09, 10.03, 10.24, 10.19, 10.18, 10.23, 9.78, 9.96, 10.05, 10.56, 10.42, 10.29, 11.89, 10.09, 10.32, 10.41, 10.28, 10.32, 10.44, 10.23, 11.00, 10.29, 10.67]
print(np.mean(real_time))
# print(len(real_time))
print(np.max(real_time))
print(np.min(real_time))
imaginary_time = [10.43, 11.10, 10.50, 10.70, 11.03, 10.72, 11.64, 10.47, 10.58, 10.35, 11.24, 11.46, 10.94, 11.14, 11.56, 13.05, 11.69, 11.29, 11.08, 12.01, 11.36, 11.50, 10.79, 10.19, 10.30, 10.36, 10.15, 10.71, 10.02, 9.87, 10.21, 10.48, 10.02, 9.96, 10.10, 10.17, 10.12, 10.13, 9.98, 10.05, 10.02, 9.92, 10.33, 10.58, 10.18, 10.32, 10.06, 9.77, 10.42, 10.36, 10.25, 10.06, 10.19, 10.01, 10.63, 10.31, 9.98, 11.33, 10.16, 9.84, 10.00, 10.40, 10.74, 10.79, 10.32, 10.77, 10.67, 10.25, 10.09, 10.32, 10.24, 10.16, 10.09, 10.07, 10.25, 10.43, 9.75, 10.78, 10.86, 10.26, 10.10, 10.13, 10.33, 10.39, 10.20, 10.13, 10.15, 10.40, 10.22, 10.87, 10.74, 10.30, 10.06, 10.19, 10.33, 10.08, 10.27, 10.60, 10.06, 10.09]
print(np.mean(imaginary_time))
# print(len(imaginary_time))
print(np.max(imaginary_time))
print(np.min(imaginary_time))

10.460500000000001
100
14.69
9.78
10.470499999999998
100
13.05
9.75


In [23]:
# # ==========================================
# # Initialization & Setup
# # ==========================================
#
# # MAKE SURE THIS PATH POINTS TO YOUR NEW .pt FILES!
# pt_directory = "./16QAMpt/"
#
# # num_workers=0 to prevent the Windows Jupyter crash!
# train_loader_real = DataLoader(
#     FastOFDMDataset(pt_directory, part='real'),
#     batch_size=None, shuffle=True, num_workers=0
# )
# train_loader_imag = DataLoader(
#     FastOFDMDataset(pt_directory, part='imag'),
#     batch_size=None, shuffle=True, num_workers=0
# )
#
# input_features = 1024
# Mod_Re_NN_raw = NNICFMapper(input_features).to(device)
# Mod_Im_NN_raw = NNICFMapper(input_features).to(device)
#
# # ==========================================
# # CPU OPTIMIZATION 2: PyTorch Compilation
# # ==========================================
# print("Compiling models to optimized C++... (This might take a moment)")
# Mod_Re_NN = torch.compile(Mod_Re_NN_raw)
# Mod_Im_NN = torch.compile(Mod_Im_NN_raw)
#
# criterion = nn.MSELoss()
# optimizer_real = optim.Adam(Mod_Re_NN.parameters(), lr=0.001)
# optimizer_imag = optim.Adam(Mod_Im_NN.parameters(), lr=0.001)

This took the following times

In [35]:
real_time = [19.25, 11.35, 11.28, 11.50, 11.06, 11.85, 12.63, 11.73, 11.70, 11.58, 11.93, 11.69, 12.06, 11.99, 11.25, 11.42, 11.16, 11.18, 11.10, 11.67, 11.68, 11.90, 11.92, 11.90, 11.11, 11.50, 11.15, 11.38, 11.69, 11.56, 11.26, 11.19, 11.10, 11.13, 11.04, 11.03, 11.25, 11.63, 11.17, 11.35, 11.30, 11.63, 11.67, 11.52, 11.50, 10.85, 11.49, 11.57, 11.43, 11.27, 11.19, 11.39, 11.66, 13.05, 12.63, 11.67, 11.19, 10.99, 11.03, 11.73, 11.25, 11.12, 11.13, 11.04, 11.05, 10.99, 11.03, 11.16, 11.15, 11.09, 11.05, 11.25, 10.99, 11.55, 11.66, 10.94, 11.01, 11.53, 11.72, 11.23, 11.78, 11.77, 11.77, 11.42, 11.77, 11.40, 10.94, 11.01, 10.85, 11.12, 11.11, 11.22, 11.02, 11.38, 11.57, 11.42, 11.46, 11.22, 11.29, 11.24]
print(np.mean(real_time))
# print(len(real_time))
print(np.max(real_time))
print(np.min(real_time))
imaginary_time = [11.39, 11.08, 11.29, 11.40, 11.65, 11.70, 11.07, 13.83, 14.29, 11.66, 11.00, 11.40, 11.17, 16.42, 14.12, 12.86, 12.46, 11.97, 11.80, 11.64, 12.72, 14.06, 14.44, 14.72, 17.92, 17.01, 12.32, 11.19, 11.53, 11.68, 11.07, 10.92, 11.13, 11.06, 10.74, 10.85, 10.89, 10.98, 11.13, 10.86, 10.99, 11.12, 11.03, 10.65, 10.85, 11.00, 11.43, 11.19, 11.37, 11.08, 11.22, 11.37, 11.03, 10.94, 10.70, 11.78, 11.14, 10.64, 11.08, 10.83, 10.72, 11.15, 10.88, 11.09, 10.91, 11.46, 11.58, 11.21, 10.94, 10.87, 12.54, 11.61, 12.37, 11.66, 11.29, 11.33, 11.28, 11.75, 11.49, 11.60, 11.44, 11.76, 11.55, 11.30, 11.61, 11.53, 11.54, 11.71, 11.13, 11.03, 11.13, 10.95, 11.08, 11.06, 11.10, 10.90, 11.35, 11.00, 11.57, 11.47]
print(np.mean(imaginary_time))
# print(len(imaginary_time))
print(np.max(imaginary_time))
print(np.min(imaginary_time))

11.4878
100
19.25
10.85
11.668
100
17.92
10.64


Not using .compile() is actually better, it also avoids the hassle of cl.exe and needing to run "x64 Native Tools Command Prompt for VS". dont know if thats the case with gpu or not tho

Cell 6: Training Loop (Real Module)

In [26]:
# * takes ~18m
epochs = 100
print("--- Starting Training for the Real NN Module ---")
# ! if using .compile(), it is expected for the first epoch to take longer
for epoch in range(epochs):
    start = time.time()
    epoch_loss = 0.0
    Mod_Re_NN.train()

    for X_batch, Y_batch in train_loader_real:
        # Transfer data to GPU if possible
        X_batch, Y_batch = X_batch.to(device, non_blocking=True), Y_batch.to(device, non_blocking=True)

        optimizer_real.zero_grad(set_to_none=True)
        predictions = Mod_Re_NN(X_batch)
        loss = criterion(predictions, Y_batch)

        loss.backward()
        optimizer_real.step()

        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(train_loader_real)
    end = time.time()
    print(f"Real Module | Epoch {epoch + 1}/{epochs} | Loss: {avg_loss:.6f} | Time: {end - start:.2f} seconds")

print("Real NN Module Training Complete!")

--- Starting Training for the Real NN Module ---
Real Module | Epoch 1/100 | Loss: 0.366354 | Time: 10.91 seconds
Real Module | Epoch 2/100 | Loss: 0.280363 | Time: 10.85 seconds
Real Module | Epoch 3/100 | Loss: 0.216269 | Time: 10.63 seconds
Real Module | Epoch 4/100 | Loss: 0.166149 | Time: 11.44 seconds
Real Module | Epoch 5/100 | Loss: 0.129619 | Time: 11.77 seconds
Real Module | Epoch 6/100 | Loss: 0.105477 | Time: 14.69 seconds
Real Module | Epoch 7/100 | Loss: 0.090828 | Time: 13.13 seconds
Real Module | Epoch 8/100 | Loss: 0.082493 | Time: 11.39 seconds
Real Module | Epoch 9/100 | Loss: 0.077997 | Time: 10.53 seconds
Real Module | Epoch 10/100 | Loss: 0.075682 | Time: 10.59 seconds
Real Module | Epoch 11/100 | Loss: 0.074548 | Time: 10.47 seconds
Real Module | Epoch 12/100 | Loss: 0.074013 | Time: 10.33 seconds
Real Module | Epoch 13/100 | Loss: 0.073786 | Time: 10.22 seconds
Real Module | Epoch 14/100 | Loss: 0.073674 | Time: 10.33 seconds
Real Module | Epoch 15/100 | Loss: 0

Cell 7: Training Loop (Imaginary Module)

In [27]:
print("--- Starting Training for the Imaginary NN Module ---")

for epoch in range(epochs):
    start = time.time()
    epoch_loss = 0.0
    Mod_Im_NN.train()

    for X_batch, Y_batch in train_loader_imag:
        # Transfer data to GPU
        X_batch, Y_batch = X_batch.to(device, non_blocking=True), Y_batch.to(device, non_blocking=True)

        optimizer_imag.zero_grad(set_to_none=True)
        predictions = Mod_Im_NN(X_batch)
        loss = criterion(predictions, Y_batch)

        loss.backward()
        optimizer_imag.step()

        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(train_loader_imag)
    end = time.time()
    print(f"Imaginary Module | Epoch {epoch + 1}/{epochs} | Loss: {avg_loss:.6f} | Time: {end - start:.2f} seconds")

print("Imaginary NN Module Training Complete!")

--- Starting Training for the Imaginary NN Module ---
Imaginary Module | Epoch 1/100 | Loss: 0.470650 | Time: 10.43 seconds
Imaginary Module | Epoch 2/100 | Loss: 0.304520 | Time: 11.10 seconds
Imaginary Module | Epoch 3/100 | Loss: 0.225447 | Time: 10.50 seconds
Imaginary Module | Epoch 4/100 | Loss: 0.175898 | Time: 10.70 seconds
Imaginary Module | Epoch 5/100 | Loss: 0.143282 | Time: 11.03 seconds
Imaginary Module | Epoch 6/100 | Loss: 0.121215 | Time: 10.72 seconds
Imaginary Module | Epoch 7/100 | Loss: 0.106072 | Time: 11.64 seconds
Imaginary Module | Epoch 8/100 | Loss: 0.095623 | Time: 10.47 seconds
Imaginary Module | Epoch 9/100 | Loss: 0.088410 | Time: 10.58 seconds
Imaginary Module | Epoch 10/100 | Loss: 0.083455 | Time: 10.35 seconds
Imaginary Module | Epoch 11/100 | Loss: 0.080077 | Time: 11.24 seconds
Imaginary Module | Epoch 12/100 | Loss: 0.077799 | Time: 11.46 seconds
Imaginary Module | Epoch 13/100 | Loss: 0.076282 | Time: 10.94 seconds
Imaginary Module | Epoch 14/100 

Cell 8: Saving the Models

In [28]:
# Save the trained models
save_dir = "./trained_models/"
os.makedirs(save_dir, exist_ok=True)
# torch.save(Mod_Re_NN.state_dict(), os.path.join(save_dir, "mod_re_weights.pth"))
torch.save(Mod_Re_NN.state_dict(), os.path.join(save_dir, "mod_re_weights_no_compile.pth"))
# torch.save(Mod_Im_NN.state_dict(), os.path.join(save_dir, "mod_im_weights.pth"))
torch.save(Mod_Im_NN.state_dict(), os.path.join(save_dir, "mod_im_weights_no_compile.pth"))
print("\nTraining complete and weights saved!")


Training complete and weights saved!


Since we already saved the models, we can comment out most of the previous code blocks